# C0003R02 PARK25 LC 1D SCAN - Mass correction only

This notebook **does not create a new Excel file**. It reads the existing
`out/C0003/C0003R02_PARK25_LC_1D_SCAN.xlsx` and updates only the three mass columns in place:
`m_plate_kg`, `m_coolant_kg`, and `mtotal`. All other columns remain unchanged.


In [6]:
# ============================================================
# Cell 1 - Imports and existing OUT-file path
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import CoolProp.CoolProp as CP
from openpyxl import load_workbook

# This notebook is intended to be placed in: cases/C0003/
PROJECT_ROOT = Path.cwd().resolve().parents[1]
OUT_DIR = PROJECT_ROOT / 'out' / 'C0003'
XLSX_PATH = OUT_DIR / 'C0003R02_PARK25_LC_1D_SCAN.xlsx'

if not XLSX_PATH.is_file():
    raise FileNotFoundError(f'Existing SCAN1 Excel not found: {XLSX_PATH}')

print(f'Excel updated in place: {XLSX_PATH}')


Excel updated in place: D:\eBATS\out\C0003\C0003R02_PARK25_LC_1D_SCAN.xlsx


In [7]:
# ============================================================
# Cell 2 - Mass-model settings from C0003R03 SCAN1
# ============================================================
# SCAN1 fixed rectangular-channel geometry
W_channel = 8.0e-3       # m
H_channel = 3.0e-3       # m
L_channel = 0.380        # m

# Corrected cold-plate outer geometry
W_plate = 0.386          # m, FIXED outer plate width
H_bottom = 2.0e-3        # m, aluminum below the channel
H_plate = H_channel + H_bottom

# Material / coolant settings
rho_plate = 2719.0       # kg/m^3
fluid = 'Water'
p_water = 101325.0       # Pa

print(f'Fixed plate width = {W_plate*1e3:.1f} mm')
print(f'Plate total height = {H_plate*1e3:.1f} mm')
print(f'Channel geometry = {W_channel*1e3:.1f} x {H_channel*1e3:.1f} x {L_channel*1e3:.1f} mm')


Fixed plate width = 386.0 mm
Plate total height = 5.0 mm
Channel geometry = 8.0 x 3.0 x 380.0 mm


In [8]:
# ============================================================
# Cell 3 - Read the existing SCAN1 table
# ============================================================
scan_df = pd.read_excel(XLSX_PATH, sheet_name='Parameter scan')

required_columns = [
    'T_water_in_C',
    'num_parallel_channels',
    'm_plate_kg',
    'm_coolant_kg',
    'mtotal',
]
missing = [c for c in required_columns if c not in scan_df.columns]
if missing:
    raise RuntimeError(f'Missing required columns in existing Excel: {missing}')

print(f'Rows read from existing Excel: {len(scan_df)}')
display(scan_df.head())


Rows read from existing Excel: 144


,case_id,T_water_in_C,m_dot_total_kg_s,num_cell_seg,num_parallel_channels,m_dot_channel_kg_s,Dh_mm,u_water_m_s,Re_water,P_pump,...,DeltaT_mission_max_C,Twater_out_cruise_end_C,dTmax_takeoff_C,dTmax_climb_C,dTmax_transition1_C,dTmax_cruise_C,dTmax_transition2_C,dTmax_descent_C,dTmax_hover_C,dTmax_landing_C
0,C0003_S001,20,0.045,1.333333,12,0.003750,4.363636,0.156531,680.731637,0.013402,...,1.085045,20.475053,0.415090,1.475418,1.115929,-0.479069,-0.755113,0.508139,12.459443,0.376525
1,C0003_S002,20,0.045,1.000000,16,0.002812,4.363636,0.117398,510.548728,0.010052,...,1.423821,20.469645,0.414166,1.453403,0.932040,-0.808965,-0.714706,0.515965,11.829531,0.339079
2,C0003_S003,20,0.045,0.800000,20,0.002250,4.363636,0.093918,408.438982,0.008041,...,1.775868,20.465828,0.413397,1.435172,0.786187,-0.964211,-0.682568,0.521732,11.317774,0.309541
3,C0003_S004,20,0.045,0.666667,24,0.001875,4.363636,0.078265,340.365818,0.006701,...,2.124363,20.463283,0.412765,1.420130,0.667734,-1.041080,-0.657209,0.526117,10.895990,0.285642
4,C0003_S005,20,0.045,0.571429,28,0.001607,4.363636,0.067085,291.742130,0.005744,...,2.458603,20.461535,0.412251,1.407779,0.569360,-1.079064,-0.636819,0.529654,10.543664,0.265857


In [9]:
# ============================================================
# Cell 4 - Recalculate mass directly from each Excel row
# ============================================================
def calculate_mass_from_row(T_water_C, num_parallel_channels):
    n_ch = int(round(float(num_parallel_channels)))
    T_water_K = float(T_water_C) + 273.15

    rho_water = float(
        CP.PropsSI('D', 'T', T_water_K, 'P', p_water, fluid)
    )

    # Channel volume: all parallel channels together
    V_channels = n_ch * W_channel * H_channel * L_channel

    # Fixed external cold-plate envelope volume
    V_plate_outer = W_plate * H_plate * L_channel

    # Aluminum volume = full plate envelope - internal channel voids
    V_aluminum = V_plate_outer - V_channels
    if V_aluminum <= 0.0:
        raise ValueError(
            f'Non-positive aluminum volume for n_ch={n_ch}: {V_aluminum}'
        )

    m_plate_kg = rho_plate * V_aluminum
    m_coolant_kg = rho_water * V_channels
    mtotal = m_plate_kg + m_coolant_kg

    return m_plate_kg, m_coolant_kg, mtotal

mass_rows = scan_df.apply(
    lambda row: calculate_mass_from_row(
        row['T_water_in_C'],
        row['num_parallel_channels'],
    ),
    axis=1,
    result_type='expand',
)
mass_rows.columns = ['m_plate_kg_new', 'm_coolant_kg_new', 'mtotal_new']

comparison = pd.concat([
    scan_df[['case_id','T_water_in_C','m_dot_total_kg_s','num_parallel_channels','m_plate_kg','m_coolant_kg','mtotal']],
    mass_rows,
], axis=1)

display(comparison.head(12))


,case_id,T_water_in_C,m_dot_total_kg_s,num_parallel_channels,m_plate_kg,m_coolant_kg,mtotal,m_plate_kg_new,m_coolant_kg_new,mtotal_new
0,C0003_S001,20,0.045,12,1.391799,0.109244,1.501043,1.696547,0.109244,1.805791
1,C0003_S002,20,0.045,16,1.825131,0.145658,1.970790,1.597358,0.145658,1.743017
2,C0003_S003,20,0.045,20,2.258464,0.182073,2.440537,1.498169,0.182073,1.680242
3,C0003_S004,20,0.045,24,2.691796,0.218488,2.910284,1.398980,0.218488,1.617467
4,C0003_S005,20,0.045,28,3.125129,0.254902,3.380031,1.299791,0.254902,1.554693
5,C0003_S006,20,0.045,32,3.558461,0.291317,3.849778,1.200602,0.291317,1.491918
6,C0003_S007,20,0.050,12,1.391799,0.109244,1.501043,1.696547,0.109244,1.805791
7,C0003_S008,20,0.050,16,1.825131,0.145658,1.970790,1.597358,0.145658,1.743017
8,C0003_S009,20,0.050,20,2.258464,0.182073,2.440537,1.498169,0.182073,1.680242
9,C0003_S010,20,0.050,24,2.691796,0.218488,2.910284,1.398980,0.218488,1.617467


In [10]:
# ============================================================
# Cell 5 - Update ONLY the mass columns in the ORIGINAL Excel
# ============================================================
# IMPORTANT:
#   No new Excel file is created.
#   The original file is overwritten in place:
#   out/C0003/C0003R02_PARK25_LC_1D_SCAN.xlsx

wb = load_workbook(XLSX_PATH)
ws = wb['Parameter scan']

headers = [cell.value for cell in ws[1]]
col = {name: idx + 1 for idx, name in enumerate(headers)}

for name in ['m_plate_kg', 'm_coolant_kg', 'mtotal']:
    if name not in col:
        raise RuntimeError(f'Missing mass column in Excel: {name}')

if ws.max_row - 1 != len(scan_df):
    raise RuntimeError(
        f'Row-count mismatch: openpyxl={ws.max_row-1}, pandas={len(scan_df)}'
    )

for i, excel_row in enumerate(range(2, ws.max_row + 1)):
    mp, mc, mt = mass_rows.iloc[i]
    ws.cell(excel_row, col['m_plate_kg']).value = float(mp)
    ws.cell(excel_row, col['m_coolant_kg']).value = float(mc)
    ws.cell(excel_row, col['mtotal']).value = float(mt)

# Save back to the SAME Excel file.
wb.save(XLSX_PATH)

print(f'Original Excel updated successfully: {XLSX_PATH}')
print(f'Updated rows: {len(scan_df)}')
print('No additional Excel file was generated.')


Original Excel updated successfully: D:\eBATS\out\C0003\C0003R02_PARK25_LC_1D_SCAN.xlsx
Updated rows: 144
No additional Excel file was generated.


In [11]:
# ============================================================
# Cell 6 - Verification
# ============================================================
check_df = pd.read_excel(XLSX_PATH, sheet_name='Parameter scan')

# Confirm all non-mass columns are unchanged.
non_mass_cols = [
    c for c in scan_df.columns
    if c not in ['m_plate_kg', 'm_coolant_kg', 'mtotal']
]

for c in non_mass_cols:
    a = scan_df[c]
    b = check_df[c]
    if pd.api.types.is_numeric_dtype(a):
        if not np.allclose(a.to_numpy(dtype=float), b.to_numpy(dtype=float), equal_nan=True):
            raise RuntimeError(f'Non-mass numeric column changed unexpectedly: {c}')
    else:
        if not a.fillna('').astype(str).equals(b.fillna('').astype(str)):
            raise RuntimeError(f'Non-mass text column changed unexpectedly: {c}')

mass_sum_error = np.max(
    np.abs(
        check_df['mtotal'].to_numpy(dtype=float)
        - check_df['m_plate_kg'].to_numpy(dtype=float)
        - check_df['m_coolant_kg'].to_numpy(dtype=float)
    )
)

print(f'Maximum mass sum error = {mass_sum_error:.3e} kg')
print('Only m_plate_kg, m_coolant_kg and mtotal were updated.')

# Representative rows for 25 degC and 0.045 kg/s
rep = check_df[
    np.isclose(check_df['T_water_in_C'], 25.0)
    & np.isclose(check_df['m_dot_total_kg_s'], 0.045)
][[
    'case_id',
    'num_parallel_channels',
    'm_plate_kg',
    'm_coolant_kg',
    'mtotal',
]]
display(rep)


Maximum mass sum error = 6.106e-16 kg
Only m_plate_kg, m_coolant_kg and mtotal were updated.


,case_id,num_parallel_channels,m_plate_kg,m_coolant_kg,mtotal
36,C0003_S037,12,1.696547,0.109117,1.805664
37,C0003_S038,16,1.597358,0.145489,1.742847
38,C0003_S039,20,1.498169,0.181861,1.680030
39,C0003_S040,24,1.398980,0.218234,1.617214
40,C0003_S041,28,1.299791,0.254606,1.554397
41,C0003_S042,32,1.200602,0.290978,1.491580
